# Classification & Regression Metrics

Companion notebook for the [Classification & Regression Metrics lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/01-classification-metrics).

**The idea in one sentence.** "Accuracy" is almost never the right number — the
metric you optimise silently defines what "good" means, and the wrong choice hides
catastrophic failures (a 99%-accurate model that never catches the 1% fraud).

What this notebook builds and validates against scikit-learn:

- **The confusion matrix** — TP/FP/FN/TN, the source of every classification metric.
- **The precision–recall tradeoff** — one threshold can't maximise both; pick per cost.
- **ROC vs PR curves** — on **imbalanced** data ROC-AUC looks rosy while PR-AUC
  tells the truth.
- **Regression metrics** — MAE vs RMSE (outlier sensitivity) and $R^2$.

Every from-scratch metric is checked against `sklearn`; the exercises re-derive
ROC-AUC and a full metrics bank.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (roc_curve, precision_recall_curve, auc,
                              roc_auc_score, average_precision_score)

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['grid.color']       = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## 1. Confusion Matrix

Generate synthetic binary predictions and visualize the confusion matrix with all four derived metrics.

In [ ]:
# Generate synthetic binary labels and scores
n = 500
y_true = (rng.uniform(0, 1, n) < 0.3).astype(int)  # 30% positive
# Scores: positives skewed high, negatives skewed low
scores = np.where(y_true == 1,
                  rng.beta(5, 2, n),   # positives: high scores
                  rng.beta(2, 5, n))   # negatives: low scores

# Apply threshold 0.5
y_pred = (scores >= 0.5).astype(int)

TP = np.sum((y_pred == 1) & (y_true == 1))
FP = np.sum((y_pred == 1) & (y_true == 0))
TN = np.sum((y_pred == 0) & (y_true == 0))
FN = np.sum((y_pred == 0) & (y_true == 1))

precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)
accuracy  = (TP + TN) / n

# Plot confusion matrix
cm = np.array([[TN, FP], [FN, TP]])
labels = [['TN', 'FP'], ['FN', 'TP']]

fig, (ax_cm, ax_metrics) = plt.subplots(1, 2, figsize=(12, 4))

im = ax_cm.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{labels[i][j]}\n{cm[i,j]}',
                   ha='center', va='center', color='white', fontsize=14, fontweight='bold')
ax_cm.set_xticks([0, 1]); ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(['Pred Neg', 'Pred Pos'])
ax_cm.set_yticklabels(['Actual Neg', 'Actual Pos'])
ax_cm.set_title('Confusion Matrix', color='white')
plt.colorbar(im, ax=ax_cm)

metrics_names  = ['Accuracy', 'Precision', 'Recall', 'F1']
metrics_values = [accuracy, precision, recall, f1]
bar_colors     = [BRAND, TEAL, YELLOW, ROSE]
bars = ax_metrics.bar(metrics_names, metrics_values, color=bar_colors, edgecolor='none')
for bar, val in zip(bars, metrics_values):
    ax_metrics.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=11)
ax_metrics.set_ylim(0, 1.1)
ax_metrics.set_title('Derived Metrics', color='white')
ax_metrics.set_ylabel('Score')
ax_metrics.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print(f'TP={TP}, FP={FP}, TN={TN}, FN={FN}')

**What to notice — the confusion matrix is everything.** Precision =
$TP/(TP+FP)$ ("of what I flagged, how much was right"), recall = $TP/(TP+FN)$ ("of
what was real, how much did I catch"). They pull in opposite directions, and
accuracy hides the tradeoff entirely on skewed classes.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
print(f'our precision {precision:.4f} vs sklearn {precision_score(y_true, y_pred):.4f}')
print(f'our recall    {recall:.4f} vs sklearn {recall_score(y_true, y_pred):.4f}')
print(f'our f1        {f1:.4f} vs sklearn {f1_score(y_true, y_pred):.4f}')
assert abs(precision - precision_score(y_true, y_pred)) < 1e-9
assert abs(recall - recall_score(y_true, y_pred)) < 1e-9
assert abs(f1 - f1_score(y_true, y_pred)) < 1e-9
print('✅ our confusion-matrix metrics match scikit-learn exactly')

## 2. Precision-Recall Tradeoff

Precision and recall trade off as we vary the classification threshold.
The F1-maximizing threshold is not always 0.5.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    yp = (scores >= t).astype(int)
    tp = np.sum((yp == 1) & (y_true == 1))
    fp = np.sum((yp == 1) & (y_true == 0))
    fn = np.sum((yp == 0) & (y_true == 1))
    p  = tp / (tp + fp + 1e-9)
    r  = tp / (tp + fn + 1e-9)
    precisions.append(p)
    recalls.append(r)
    f1s.append(2 * p * r / (p + r + 1e-9))

best_t_idx = np.argmax(f1s)
best_t     = thresholds[best_t_idx]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, precisions, color=TEAL,   linewidth=2, label='Precision')
ax.plot(thresholds, recalls,    color=YELLOW,  linewidth=2, label='Recall')
ax.plot(thresholds, f1s,        color=BRAND,   linewidth=2, label='F1', linestyle='--')
ax.axvline(best_t, color=ROSE, linestyle=':', linewidth=1.5,
           label=f'Best F1 threshold = {best_t:.2f}')
ax.set_xlabel('Classification threshold')
ax.set_ylabel('Score')
ax.set_title('Precision-Recall Tradeoff vs. Threshold', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best threshold: {best_t:.2f} → F1={max(f1s):.3f}')

**What to notice — the threshold is a business decision.** Raising the
threshold lifts precision and drops recall; lowering it does the reverse. There is
no universally best threshold — a cancer screen wants high recall, a spam filter
high precision. F1 picks the balance point only if the two errors cost the same.

## 3. ROC vs PR Curves on Imbalanced Data

With only 5% positive rate, the ROC curves for two classifiers look similar,
but the PR curves reveal that Classifier B is much worse in practice.

In [ ]:
rng2 = np.random.default_rng(7)
n2 = 2000
y2 = (rng2.uniform(0, 1, n2) < 0.05).astype(int)  # 5% positive

# Good classifier
scores_a = np.where(y2 == 1, rng2.beta(8, 2, n2), rng2.beta(2, 8, n2))
# Worse classifier (less separation)
scores_b = np.where(y2 == 1, rng2.beta(4, 3, n2), rng2.beta(3, 4, n2))

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))

for scores, label, color in [(scores_a, 'Classifier A', TEAL), (scores_b, 'Classifier B', ROSE)]:
    fpr, tpr, _ = roc_curve(y2, scores)
    roc_auc = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, color=color, linewidth=2, label=f'{label} (AUC={roc_auc:.3f})')

    prec, rec, _ = precision_recall_curve(y2, scores)
    pr_auc = auc(rec, prec)
    ax_pr.plot(rec, prec, color=color, linewidth=2, label=f'{label} (AUC={pr_auc:.3f})')

ax_roc.plot([0,1],[0,1],'--', color='gray', alpha=0.5, label='Random (AUC=0.5)')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves (looks similar!)', color='white')
ax_roc.legend(); ax_roc.grid(True, alpha=0.3)

ax_pr.axhline(y=0.05, color='gray', linestyle='--', alpha=0.5, label='No-skill (prevalence=5%)')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('PR Curves (difference is clear!)', color='white')
ax_pr.legend(); ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What to notice — ROC lies on imbalanced data.** Both classifiers look
excellent on the ROC curve (high AUC), because the true-negative-rich denominator
of FPR barely moves. The **PR curve** exposes the real gap: with 5% positives,
precision is what a user actually feels, and Classifier B collapses. On imbalanced
problems, **prefer PR-AUC**.

## 4. Regression Metrics

MAE, MSE, RMSE, and R² on a noisy prediction problem. See how outliers affect MSE disproportionately.

In [ ]:
rng3 = np.random.default_rng(13)
x_reg = np.linspace(0, 10, 100)
y_true_reg = np.sin(x_reg) + 0.5 * x_reg

# Add noise and a few outliers
noise = rng3.normal(0, 0.3, 100)
outlier_idx = [20, 50, 80]
noise[outlier_idx] = rng3.normal(0, 3, 3)  # large outlier errors
y_pred_reg = y_true_reg + noise

residuals = y_pred_reg - y_true_reg
mae  = np.mean(np.abs(residuals))
mse  = np.mean(residuals**2)
rmse = np.sqrt(mse)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_true_reg - y_true_reg.mean())**2)
r2 = 1 - ss_res / ss_tot

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(x_reg, y_true_reg, color=TEAL, linewidth=2, label='True')
ax1.scatter(x_reg, y_pred_reg, color=BRAND, s=15, alpha=0.7, label='Predicted')
for idx in outlier_idx:
    ax1.annotate('outlier', (x_reg[idx], y_pred_reg[idx]), fontsize=8,
                 color=ROSE, xytext=(x_reg[idx]+0.3, y_pred_reg[idx]+0.3))
ax1.set_title(f'Predictions  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}', color='white')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.hist(residuals, bins=25, color=BRAND, alpha=0.7, edgecolor='none')
ax2.axvline(0, color=TEAL, linewidth=1.5, linestyle='--')
ax2.set_xlabel('Residual (y_pred - y_true)')
ax2.set_title('Residual distribution (outliers visible as long tail)', color='white')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'MAE  = {mae:.4f}  (average absolute error)')
print(f'MSE  = {mse:.4f}  (average squared error — outliers dominate)')
print(f'RMSE = {rmse:.4f}  (same units as y)')
print(f'R²   = {r2:.4f}  (fraction of variance explained)')

**What to notice — MAE vs RMSE and the outliers.** RMSE squares the
residuals, so the three injected outliers dominate it; MAE weights every error
linearly and barely moves. Report RMSE when large errors are especially bad, MAE
when you want a robust typical-error, and $R^2$ for variance-explained — but never
just one.

---
## ✏️ Your turn

Each exercise has a stub to fill in, an assert cell that prints `✅` when correct, and a solution in `<details>`.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **accuracy on imbalance** | a majority-class predictor scores high while catching nothing (demo) |
| **single threshold** | precision and recall trade off; choose the threshold by error cost, not F1 by default |
| **ROC-AUC on rare positives** | looks great because FPR barely moves; use PR-AUC instead |
| **RMSE vs MAE** | RMSE is outlier-dominated; MAE is robust — they answer different questions |
| **$R^2$ can be negative** | a model worse than the mean predictor gives $R^2<0$ |

Demo: accuracy is useless on imbalanced data.

In [ ]:
# Accuracy is dangerous on imbalanced data: a model that ALWAYS predicts 'negative'
# scores high accuracy while catching zero positives. Precision/recall expose it.
y_imb = (rng.uniform(0, 1, 2000) < 0.02).astype(int)   # 2% positive
always_negative = np.zeros_like(y_imb)
acc = (always_negative == y_imb).mean()
tp = np.sum((always_negative == 1) & (y_imb == 1))
rec = tp / max(y_imb.sum(), 1)
print(f'"always predict negative" -> accuracy {acc:.1%}, recall {rec:.1%}')
assert acc > 0.95 and rec == 0.0
print('98% accurate and completely useless. This is why accuracy alone is a trap on skewed data.')

### Exercise 1 — `compute_metrics(y_true, y_pred)`

Compute precision, recall, F1, and accuracy from scratch (no sklearn).

Return a dict `{'precision': ..., 'recall': ..., 'f1': ..., 'accuracy': ...}`.

Formulas:
- $\text{Precision} = TP / (TP + FP)$
- $\text{Recall} = TP / (TP + FN)$
- $F_1 = 2 \cdot P \cdot R / (P + R)$
- $\text{Accuracy} = (TP + TN) / N$

In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Compute classification metrics from binary labels.
    Returns dict with precision, recall, f1, accuracy.
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    # TODO(you): compute TP, FP, TN, FN then derive the four metrics
    TP = ...
    FP = ...
    TN = ...
    FN = ...
    precision = ...
    recall    = ...
    f1        = ...
    accuracy  = ...
    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}

In [ ]:
from sklearn import metrics as sk_metrics

yt = rng.integers(0, 2, 200)
yp = rng.integers(0, 2, 200)

result = compute_metrics(yt, yp)
assert result is not None, "compute_metrics returned None"
assert abs(result['precision'] - sk_metrics.precision_score(yt, yp, zero_division=0)) < 1e-9, \
    f"Precision mismatch: {result['precision']:.4f} vs {sk_metrics.precision_score(yt, yp, zero_division=0):.4f}"
assert abs(result['recall'] - sk_metrics.recall_score(yt, yp, zero_division=0)) < 1e-9, \
    f"Recall mismatch"
assert abs(result['f1'] - sk_metrics.f1_score(yt, yp, zero_division=0)) < 1e-9, \
    f"F1 mismatch"
assert abs(result['accuracy'] - sk_metrics.accuracy_score(yt, yp)) < 1e-9, \
    f"Accuracy mismatch"

# Edge cases (in the spirit of DML's tests.json style)
all_correct = compute_metrics([1, 0, 1, 1, 0], [1, 0, 1, 1, 0])
assert abs(all_correct['accuracy'] - 1.0) < 1e-9 and abs(all_correct['f1'] - 1.0) < 1e-9, \
    "All-correct predictions should give accuracy=1.0 and f1=1.0"

all_wrong = compute_metrics([1, 0, 1, 1, 0], [0, 1, 0, 0, 1])
assert abs(all_wrong['accuracy']) < 1e-9 and abs(all_wrong['precision']) < 1e-9 and abs(all_wrong['recall']) < 1e-9, \
    "All-wrong predictions should give accuracy=precision=recall=0.0"

all_negative = compute_metrics([0, 0, 0, 0], [0, 0, 0, 0])
assert abs(all_negative['accuracy'] - 1.0) < 1e-9 and abs(all_negative['precision']) < 1e-9 and abs(all_negative['recall']) < 1e-9, \
    "All-one-class (negative) labels with no predicted positives: accuracy=1.0, precision=recall=0.0 by convention"

print("\u2705 Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compute_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    TP = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / len(y_true)
    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}
```

</details>

### Exercise 2 — `roc_auc_from_scratch(y_true, y_scores)`

Compute AUC-ROC from scratch without sklearn:
1. Sort predictions by score descending
2. Walk the sorted list, accumulating TPR and FPR at each threshold
3. Compute the area under this curve via the trapezoid rule

**Hint:** at each position, count cumulative TP and FP.

In [ ]:
def roc_auc_from_scratch(y_true, y_scores):
    """
    Compute ROC AUC without sklearn.
    Returns scalar AUC in [0, 1].
    """
    y_true, y_scores = np.asarray(y_true), np.asarray(y_scores)
    # TODO(you): sort by score descending, trace TPR vs FPR, compute trapezoid area
    # Hint: sort_idx = np.argsort(y_scores)[::-1]
    #       then iterate accumulating TP and FP counts
    #       TPR = cumulative_TP / total_positives
    #       FPR = cumulative_FP / total_negatives
    return ...

In [ ]:
# Test on the scores from earlier
my_auc_a = roc_auc_from_scratch(y2, scores_a)
my_auc_b = roc_auc_from_scratch(y2, scores_b)
sk_auc_a = roc_auc_score(y2, scores_a)
sk_auc_b = roc_auc_score(y2, scores_b)

assert my_auc_a is not None, "roc_auc_from_scratch returned None"
assert abs(my_auc_a - sk_auc_a) < 0.01, \
    f"Classifier A: expected {sk_auc_a:.4f}, got {my_auc_a:.4f}"
assert abs(my_auc_b - sk_auc_b) < 0.01, \
    f"Classifier B: expected {sk_auc_b:.4f}, got {my_auc_b:.4f}"
assert my_auc_a > my_auc_b, "Classifier A should have higher AUC"

# Edge case: perfectly separated scores should give AUC = 1.0
y_perfect = np.array([0] * 50 + [1] * 50)
scores_perfect = np.arange(100)  # negatives all score lower than positives
auc_perfect = roc_auc_from_scratch(y_perfect, scores_perfect)
assert abs(auc_perfect - 1.0) < 1e-9, f"Perfect separation should give AUC=1.0, got {auc_perfect}"

print(f"AUC A: yours={my_auc_a:.4f}, sklearn={sk_auc_a:.4f}")
print(f"AUC B: yours={my_auc_b:.4f}, sklearn={sk_auc_b:.4f}")
print("\u2705 Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def roc_auc_from_scratch(y_true, y_scores):
    y_true, y_scores = np.asarray(y_true), np.asarray(y_scores)
    sort_idx = np.argsort(y_scores)[::-1]
    y_sorted = y_true[sort_idx]
    total_pos = y_true.sum()
    total_neg = len(y_true) - total_pos
    tprs = [0.0]
    fprs = [0.0]
    tp, fp = 0, 0
    for label in y_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1
        tprs.append(tp / total_pos)
        fprs.append(fp / total_neg)
    return float((np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(tprs, fprs))
```

</details>

### Exercise 3 — Classification-metrics bank (DML 36, 46, 52, 61, 91, 72, 73, 75, 77)

Open-Deep-ML's coding-problem bank has nine separate binary-classification
metric problems that all reduce to the same four counts. Implement
`confusion_matrix_counts` once, then derive every metric from it:

- DML `36` `calculate-accuracy-score` → `accuracy_score`
- DML `46` `implement-precision-metric` → `precision_score`
- DML `52` `implement-recall-metric-in-binary-classification` → `recall_score`
- DML `61` `implement-f-score-calculation-for-binary-classific` → `f_beta_score(y_true, y_pred, beta)`
- DML `91` `calculate-f1-score-from-predicted-and-true-labels` → `f_beta_score(..., beta=1.0)`
- DML `72` `calculate-jaccard-index-for-binary-classification` → `jaccard_index`
- DML `73` `calculate-dice-score-for-classification` → `dice_score`
- DML `75` `generate-a-confusion-matrix-for-binary-classificat` → `confusion_matrix_from_pairs` (DML's `[y_true, y_pred]` pair-list input format)
- DML `77` `calculate-performance-metrics-for-a-classification` → `performance_metrics` (confusion matrix + accuracy + F1 + specificity + NPV in one call)

All counts and ratios follow DML's convention: confusion matrix layout
`[[TP, FN], [FP, TN]]`, and every ratio guards against a zero denominator by
returning `0.0` instead of raising. (Fun fact you can check below: for binary
labels, the Dice score and the F1 score are the *same* formula —
`2*TP / (2*TP + FP + FN)` is algebraically identical to `2*P*R / (P+R)`.)

In [ ]:
def confusion_matrix_counts(y_true, y_pred):
    """
    Shared TP/FP/FN/TN counts used by every metric below.
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    # TODO(you): compute the four confusion-matrix counts
    TP = ...
    FP = ...
    FN = ...
    TN = ...
    return TP, FP, FN, TN


def accuracy_score(y_true, y_pred):            # DML 36
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    # TODO(you): (TP + TN) / total
    return ...


def precision_score(y_true, y_pred):           # DML 46
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    # TODO(you): TP / (TP + FP), guarding TP + FP == 0 -> 0.0
    return ...


def recall_score(y_true, y_pred):              # DML 52
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    # TODO(you): TP / (TP + FN), guarding TP + FN == 0 -> 0.0
    return ...


def f_beta_score(y_true, y_pred, beta=1.0):    # DML 61 (general beta), DML 91 (beta=1 -> F1)
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    # TODO(you): F_beta = (1 + beta**2) * p * r / (beta**2 * p + r)
    # guard the denominator == 0 -> 0.0, and round the result to 3 decimal places
    return ...


def jaccard_index(y_true, y_pred):             # DML 72
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    # TODO(you): TP / (TP + FP + FN), guard denom == 0 -> 0.0, round to 3 decimals
    return ...


def dice_score(y_true, y_pred):                # DML 73
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    # TODO(you): 2*TP / (2*TP + FP + FN), guard denom == 0 -> 0.0, round to 3 decimals
    return ...


def confusion_matrix_from_pairs(data):         # DML 75 -- data is a list of [y_true, y_pred] pairs
    # TODO(you): split `data` into y_true / y_pred lists, reuse confusion_matrix_counts,
    # and return [[TP, FN], [FP, TN]]
    return ...


def performance_metrics(y_true, y_pred):       # DML 77
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    # TODO(you): return (confusion_matrix, accuracy, f1, specificity, npv), each
    # rounded to 3 decimal places except confusion_matrix (which holds ints).
    # confusion_matrix = [[TP, FN], [FP, TN]]
    # specificity = TN / (TN + FP), guard denom == 0 -> 0.0
    # npv         = TN / (TN + FN), guard denom == 0 -> 0.0
    return ...

In [ ]:
import math

def close(a, b, tol=1e-9):
    return math.isclose(a, b, abs_tol=tol)

# Shared test vectors (matches DML 77's own first test case)
yt = [1, 0, 1, 0, 1]
yp = [1, 0, 0, 1, 1]

TP, FP, FN, TN = confusion_matrix_counts(yt, yp)
assert (TP, FP, FN, TN) == (2, 1, 1, 1), f"Expected counts (2,1,1,1), got {(TP, FP, FN, TN)}"

assert close(accuracy_score(yt, yp), 0.6), f"accuracy_score: {accuracy_score(yt, yp)}"
assert close(precision_score(yt, yp), 2 / 3), f"precision_score: {precision_score(yt, yp)}"
assert close(recall_score(yt, yp), 2 / 3), f"recall_score: {recall_score(yt, yp)}"
assert close(f_beta_score(yt, yp, beta=1.0), 0.667), f"f_beta_score(beta=1): {f_beta_score(yt, yp, beta=1.0)}"
assert close(f_beta_score(yt, yp, beta=2.0), 0.667), f"f_beta_score(beta=2): {f_beta_score(yt, yp, beta=2.0)}"
assert close(jaccard_index(yt, yp), 0.5), f"jaccard_index: {jaccard_index(yt, yp)}"
assert close(dice_score(yt, yp), 0.667), f"dice_score: {dice_score(yt, yp)}"
assert close(dice_score(yt, yp), f_beta_score(yt, yp, beta=1.0)), "Dice and F1 should be numerically identical here"

# DML 75's own example: pair-list input format
assert confusion_matrix_from_pairs([[1, 1], [1, 0], [0, 1], [0, 0], [0, 1]]) == [[1, 1], [2, 1]], \
    "confusion_matrix_from_pairs mismatch on DML 75's example"

# DML 77's own example: full performance_metrics tuple
pm = performance_metrics([1, 0, 1, 0, 1], [1, 0, 0, 1, 1])
assert pm[0] == [[2, 1], [1, 1]], f"confusion matrix mismatch: {pm[0]}"
assert close(pm[1], 0.6) and close(pm[2], 0.667) and close(pm[3], 0.5) and close(pm[4], 0.5), \
    f"performance_metrics mismatch: {pm}"

# Edge case: all-correct predictions
yt_c, yp_c = [1, 0, 1, 1, 0], [1, 0, 1, 1, 0]
assert close(accuracy_score(yt_c, yp_c), 1.0)
assert close(f_beta_score(yt_c, yp_c, beta=1.0), 1.0)
assert close(jaccard_index(yt_c, yp_c), 1.0)
assert close(dice_score(yt_c, yp_c), 1.0)

# Edge case: all-wrong predictions
yt_w, yp_w = [1, 0, 1, 1, 0], [0, 1, 0, 0, 1]
assert close(accuracy_score(yt_w, yp_w), 0.0)
assert close(precision_score(yt_w, yp_w), 0.0)
assert close(recall_score(yt_w, yp_w), 0.0)
assert close(jaccard_index(yt_w, yp_w), 0.0)
assert close(dice_score(yt_w, yp_w), 0.0)

# Edge case: all-one-class labels (true is entirely negative, nothing predicted positive)
yt_neg, yp_neg = [0, 0, 0, 0], [0, 0, 0, 0]
assert close(accuracy_score(yt_neg, yp_neg), 1.0)
assert close(precision_score(yt_neg, yp_neg), 0.0), "0/0 precision should be defined as 0.0, not raise"
assert close(recall_score(yt_neg, yp_neg), 0.0)
assert close(jaccard_index(yt_neg, yp_neg), 0.0), "0/0 Jaccard should be defined as 0.0 (matches DML 73's own zero/zero test)"
assert close(dice_score(yt_neg, yp_neg), 0.0)

# Edge case: all-one-class labels (true is entirely positive)
yt_pos, yp_pos = [1, 1, 1, 1], [1, 0, 1, 0]
assert close(precision_score(yt_pos, yp_pos), 1.0)
assert close(recall_score(yt_pos, yp_pos), 0.5)

print("\u2705 Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def confusion_matrix_counts(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    TP = int(np.sum((y_pred == 1) & (y_true == 1)))
    FP = int(np.sum((y_pred == 1) & (y_true == 0)))
    FN = int(np.sum((y_pred == 0) & (y_true == 1)))
    TN = int(np.sum((y_pred == 0) & (y_true == 0)))
    return TP, FP, FN, TN


def accuracy_score(y_true, y_pred):
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    return (TP + TN) / (TP + FP + FN + TN)


def precision_score(y_true, y_pred):
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0


def recall_score(y_true, y_pred):
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0


def f_beta_score(y_true, y_pred, beta=1.0):
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    denom = beta ** 2 * p + r
    return round((1 + beta ** 2) * p * r / denom, 3) if denom > 0 else 0.0


def jaccard_index(y_true, y_pred):
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    denom = TP + FP + FN
    return round(TP / denom, 3) if denom > 0 else 0.0


def dice_score(y_true, y_pred):
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    denom = 2 * TP + FP + FN
    return round(2 * TP / denom, 3) if denom > 0 else 0.0


def confusion_matrix_from_pairs(data):
    y_true = [pair[0] for pair in data]
    y_pred = [pair[1] for pair in data]
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    return [[TP, FN], [FP, TN]]


def performance_metrics(y_true, y_pred):
    TP, FP, FN, TN = confusion_matrix_counts(y_true, y_pred)
    confusion = [[TP, FN], [FP, TN]]
    accuracy = (TP + TN) / (TP + FP + FN + TN)
    f1 = f_beta_score(y_true, y_pred, beta=1.0)
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0.0
    return confusion, round(accuracy, 3), f1, round(specificity, 3), round(npv, 3)
```

</details>

## Key takeaways

- **The metric defines "good."** Accuracy hides failure on imbalanced data — a
  do-nothing model can score 98% (demo).
- **Precision vs recall is a threshold tradeoff** set by the *costs* of your two
  error types, not by the model.
- **On imbalanced data, trust PR-AUC over ROC-AUC** — ROC's true-negative-heavy
  denominator flatters weak classifiers.
- **Regression: MAE is robust, RMSE punishes outliers, $R^2$ is variance
  explained** — report the one that matches what a big error costs you.
- Every from-scratch metric here matched `sklearn`; the exercises rebuild ROC-AUC
  and the full metric bank.